In [0]:
%sql
CREATE OR REPLACE TABLE customer_360.gold.customer_full_summary AS

SELECT
  customer_id,
  full_name,
  city,
  credit_score,
  credit_category,
  customer_since_year,
  customer_tenure_days,
  total_balance,
  savings_balance,
  checking_balance,
  total_loan_amount,
  avg_interest_rate,
  total_products,
  active_cards,
  avg_monthly_spending,
  engagement_score,
  churn_risk_score,
  churn_risk_tier,
  customer_value_tier,
  mobile_pct,
  total_sessions,
  total_transactions,
  successful_transactions,
  days_since_last_transaction,
  missed_payment_pct,
  avg_payment_completion_pct,
  total_interactions,
  branch_visits,
  complaints,
  notification_open_rate,
  notification_click_rate,
  is_high_value,
  is_digital_first,
  is_low_churn_risk,
  last_transaction_date,

  CONCAT(
    '👤 CUSTOMER PROFILE — ', UPPER(full_name), '\n',
    '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n',

    '📋 IDENTITY\n',
    '• Customer ID   : ', customer_id, '\n',
    '• Full Name     : ', full_name, '\n',
    '• City          : ', city, '\n',
    '• Member Since  : ', customer_since_year,
      ' (', customer_tenure_days, ' days)\n',
    '• Value Tier    : ', customer_value_tier, '\n',
    '• Credit Score  : ', credit_score, ' — ', credit_category, '\n\n',

    '💰 FINANCIAL POSITION\n',
    '• Total Balance : $', FORMAT_NUMBER(total_balance, 2), '\n',
    '  ↳ Savings     : $', FORMAT_NUMBER(savings_balance, 2), '\n',
    '  ↳ Checking    : $', FORMAT_NUMBER(checking_balance, 2), '\n',
    '• Loan Exposure : $', FORMAT_NUMBER(total_loan_amount, 2),
      ' @ ', avg_interest_rate, '% p.a.\n',
    '• Monthly Spend : $', FORMAT_NUMBER(avg_monthly_spending, 2),
      ' avg per month\n',
    '• Products Held : ', total_products, ' active products\n',
    '• Active Cards  : ', active_cards, '\n\n',

    '📊 ENGAGEMENT & BEHAVIOUR\n',
    '• Engagement Score  : ', engagement_score, ' / 100\n',
    '• Churn Risk Score  : ', churn_risk_score, ' / 100 (',
      churn_risk_tier, ' risk)\n',
    '• Digital First     : ',
      CASE WHEN is_digital_first THEN 'Yes ✓' ELSE 'No' END, '\n',
    '• Mobile Usage      : ', mobile_pct, '% of sessions\n',
    '• Total Sessions    : ', total_sessions, '\n',
    '• Total Transactions: ', total_transactions,
      ' (', successful_transactions, ' successful)\n',
    '• Last Transaction  : ', COALESCE(CAST(last_transaction_date AS STRING),
      'No transaction data'), '\n',
    '• Days Inactive     : ',
      CASE WHEN days_since_last_transaction = 999
           THEN 'No transaction history'
           ELSE CONCAT(days_since_last_transaction, ' days')
      END, '\n\n',

    '💳 PAYMENT HEALTH\n',
    '• Payment Completion: ', avg_payment_completion_pct, '%\n',
    '• Missed Payments   : ', missed_payment_pct, '% of total\n',
    '• Payment Status    : ',
      CASE
        WHEN missed_payment_pct = 0   THEN 'Excellent — no missed payments ✓'
        WHEN missed_payment_pct < 5   THEN 'Good — minimal missed payments'
        WHEN missed_payment_pct < 15  THEN 'Fair — some payment concerns'
        ELSE                               'Poor — high missed payment rate ⚠'
      END, '\n\n',

    '📞 INTERACTIONS & SERVICE\n',
    '• Total Interactions: ', total_interactions, '\n',
    '• Branch Visits     : ', branch_visits, '\n',
    '• Complaints Filed  : ', complaints,
      CASE WHEN complaints > 3 THEN ' ⚠ Review required' ELSE '' END, '\n',
    '• Notification Open : ', notification_open_rate, '%\n',
    '• Notification Click: ', notification_click_rate, '%\n\n',


    CASE
      WHEN churn_risk_score >= 60 OR missed_payment_pct > 15 OR complaints > 3
      THEN CONCAT(
        '🚨 RISK FLAGS\n',
        CASE WHEN churn_risk_score >= 60
             THEN '• HIGH CHURN RISK — immediate relationship manager action needed\n'
             ELSE '' END,
        CASE WHEN missed_payment_pct > 15
             THEN '• HIGH MISSED PAYMENT RATE — credit review recommended\n'
             ELSE '' END,
        CASE WHEN complaints > 3
             THEN '• MULTIPLE COMPLAINTS — service recovery plan needed\n'
             ELSE '' END,
        '\n')
      ELSE '✅ NO CRITICAL RISK FLAGS\n\n'
    END,

    '🎯 RECOMMENDED ACTIONS FOR BANKER\n',
    CASE
      WHEN churn_risk_score >= 60
      THEN '1. URGENT: Call customer within 24 hours — high churn risk\n'
      WHEN days_since_last_transaction > 90
      THEN '1. Re-engage: Customer inactive for 90+ days — send offer\n'
      ELSE '1. Relationship check-in: Schedule quarterly review\n'
    END,
    CASE
      WHEN missed_payment_pct > 10
      THEN '2. Payment review: Discuss restructuring options\n'
      WHEN total_loan_amount = 0 AND credit_score >= 670
      THEN '2. Cross-sell: Strong mortgage candidate (good credit, no loan)\n'
      ELSE '2. Upsell opportunity: Review product fit\n'
    END,
    CASE
      WHEN mobile_pct > 50
      THEN '3. Digital: Offer premium mobile banking features\n'
      WHEN branch_visits > 5
      THEN '3. Branch: Schedule in-person wealth review meeting\n'
      ELSE '3. Engagement: Enroll in loyalty rewards program\n'
    END,
    '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━'
  )                                                      AS customer_summary_text,

  current_timestamp()                                    AS generated_at

FROM customer_360.gold.customer_360;

In [0]:
%sql
SELECT customer_summary_text
FROM customer_360.gold.customer_full_summary
WHERE customer_id = (
  SELECT customer_id FROM customer_360.gold.customer_full_summary
  WHERE total_transactions > 0 LIMIT 1
);